# Module 14: Capstone Challenge — SOLUTION

**Snowpark ML Fundamentals — Weeks 1–3 Synthesis**

> **Instructor Note**: This is the **solution notebook** for the capstone
> challenge. Every `# TODO` has been filled in with the correct code and
> marked with `# SOLUTION`. Expert explanations precede each solution to
> highlight production best practices and the reasoning behind each step.
> The original challenge notebook is `14_capstone_challenge.ipynb`.

---

## Mission

Build and deploy a **customer risk scoring model** end-to-end using
everything covered across all three weeks:

| Week | Skills Applied |
|------|---------------|
| 1 | Data exploration, preprocessing, model training, evaluation |
| 2 | Feature Store entities, feature views, training sets |
| 3 | Model Registry versioning, alias promotion, SQL inference, monitoring |

---

## Part A: Setup & Data Exploration (10 min)

**Skills**: Session creation, data generation, DataFrame exploration (Week 1)

In [1]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../src')

import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.loader import split_data, explore_dataframe
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.feature_store.entities import (
    setup_feature_store, create_customer_entity, register_entity,
)
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
    create_rfm_features,
    create_behavioral_features,
)
from snowpark_fundamentals.feature_store.feature_views import (
    create_external_feature_view, register_feature_view, get_feature_view,
)
from snowpark_fundamentals.feature_store.training_sets import (
    create_spine_dataframe, generate_training_set, retrieve_feature_values,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    list_versions, compare_model_versions,
    set_model_alias, get_model_by_alias,
    set_model_comment, set_default_version,
    get_model_metrics,
)

session = create_session(env_path='../.env')
print("Session created.")

Session created.


### Solution A1: Generate Transaction Data

> **Key Concept**: `create_customer_transactions_dataset()` writes synthetic
> rows directly into a Snowflake table and returns a DataFrame handle. The
> heavy lifting (random generation, inserts) happens **server-side** in the
> warehouse — your local machine only receives the DataFrame reference.

In production, this step would be replaced by pointing at your actual
transactions table. The factory function here mirrors that pattern: you get
back a lazy DataFrame you can filter, join, and aggregate without pulling
data locally.

In [2]:
# TODO A1: Generate 50,000 synthetic customer transactions
# Hint: See notebook 07 / 12, cell 4 — use create_customer_transactions_dataset()

transactions_df = create_customer_transactions_dataset(session, n_rows=50000)  # SOLUTION

print(f"Transactions: {transactions_df.count()} rows")

Transactions: 50000 rows


### Solution A2: Generate Interaction Data

> **Key Concept**: Interactions (page views, clicks, support tickets) live in
> a separate table from transactions. Keeping them separate follows the
> **single-responsibility principle** for data sources — each table captures
> one behavioral domain.

We generate more interaction rows (100k) than transaction rows (50k) because
customers typically have many more micro-interactions (clicks, page views)
than purchases. This reflects real-world data volume ratios.

In [3]:
# TODO A2: Generate 100,000 synthetic customer interactions
# Hint: Same pattern as A1 — use create_customer_interactions_dataset()

interactions_df = create_customer_interactions_dataset(session, n_rows=100000)  # SOLUTION

print(f"Interactions: {interactions_df.count()} rows")

Interactions: 100000 rows


### Solution A3: Explore the Dataset

> **Key Concept**: `explore_dataframe()` returns a lightweight profile dict
> with `row_count`, `column_count`, `columns` (a list of column name strings),
> and `dtypes`. This is your first sanity check — run it immediately after
> loading data to verify schema and volume before building features.

**Bug fix**: The original challenge cell used
`[c['name'] for c in profile['columns']]`, which would fail because
`profile['columns']` is already a `list[str]`, not a `list[dict]`.
The correct code prints `profile['columns']` directly.

In [4]:
# TODO A3: Explore the transactions dataset
# Hint: See notebook 01, section 1.3 — use explore_dataframe()
# This should return a dict with keys: 'row_count', 'column_count', 'columns'

profile = explore_dataframe(transactions_df)  # SOLUTION

print(f"Rows: {profile['row_count']}")
print(f"Columns: {profile['column_count']}")
print(f"Column names: {profile['columns']}")  # SOLUTION: fixed bug — columns is list[str], not list[dict]

Rows: 50000
Columns: 8
Column names: ['TRANSACTION_ID', 'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_AMOUNT', 'CATEGORY', 'ORDER_STATUS', 'ITEM_COUNT', 'CHANNEL']


In [5]:
# --- Validation A ---
assert transactions_df.count() > 0, "A1: transactions_df should have rows"
assert interactions_df.count() > 0, "A2: interactions_df should have rows"
assert 'row_count' in profile, "A3: profile should have 'row_count' key"
print("Part A: All validations passed!")

Part A: All validations passed!


---

## Part B: Feature Engineering via Feature Store (15 min)

**Skills**: Feature computation, entity registration, feature views, training sets (Week 2)

You will:
1. Compute RFM and behavioral features from the raw data
2. Register them in the Feature Store
3. Generate a training set by joining both feature views

### Solution B1: RFM Features

> **Key Concept**: **RFM (Recency, Frequency, Monetary)** is a classic
> customer segmentation framework. `create_rfm_features()` computes these
> metrics at multiple time windows (30d, 90d, 365d, all-time) using
> server-side SQL aggregations. The result is one row per customer with
> ~12 numeric features.

Time-windowed features are critical because a customer who spent \$10k last
year but nothing in the last 30 days looks very different from one who spent
\$10k last month. Multiple windows let the model capture **trend signals**.

In [6]:
# TODO B1: Compute RFM features (recency, frequency, monetary)
# Hint: See notebook 07, section 7.3 — use create_rfm_features(session)

rfm_df = create_rfm_features(session)  # SOLUTION

print(f"RFM features: {rfm_df.count()} customers")

RFM features: 2000 customers


### Solution B2: Behavioral Features

> **Key Concept**: Behavioral features capture **engagement signals** —
> page views, clicks, support tickets, email opens. These complement RFM
> features by measuring *how* customers interact, not just *what* they buy.

In churn/risk models, behavioral features are often the strongest early
warning signals. A customer who stops clicking emails or starts filing
support tickets may churn weeks before their purchase frequency drops.

In [7]:
# TODO B2: Compute behavioral features (page views, clicks, support tickets, etc.)
# Hint: See notebook 07, section 7.4 — use create_behavioral_features(session)

behavioral_df = create_behavioral_features(session)  # SOLUTION

print(f"Behavioral features: {behavioral_df.count()} customers")

Behavioral features: 2000 customers


### Solution B3: Feature Store Setup

> **Key Concept**: The Feature Store provides **training-serving consistency**.
> By registering features once, you guarantee that the exact same
> transformations used during training are applied at inference time.

The three-step pattern — **initialize store, create entity, register views** —
is the foundation of every Feature Store workflow. The `customer_entity`
defines the join key (`CUSTOMER_ID`) that ties all feature views together.
Setting `overwrite=True` on registration makes the cell idempotent, so you
can re-run without errors.

In [8]:
# TODO B3: Set up Feature Store — 3 steps:
#   1. Initialize the Feature Store
#   2. Create and register a customer entity
#   3. Create and register two feature views (RFM + behavioral)
#
# Hint: See notebook 08, sections 8.1–8.3 and notebook 12, cell 4
#   - setup_feature_store(session)
#   - create_customer_entity(desc=...)
#   - register_entity(fs, entity)
#   - create_external_feature_view(name=..., entities=[entity], feature_df=...,
#       desc=..., timestamp_col='_FEATURE_TIMESTAMP')
#   - register_feature_view(fs, fv, version='V1', overwrite=True)

# Step 1: Initialize Feature Store
fs = setup_feature_store(session)  # SOLUTION

# Step 2: Create and register customer entity
customer_entity = create_customer_entity(desc="Customer entity for capstone")  # SOLUTION
customer_entity = register_entity(fs, customer_entity)  # SOLUTION

# Step 3a: Create and register RFM feature view
rfm_fv = create_external_feature_view(
    name="CAPSTONE_RFM_FV",
    entities=[customer_entity],
    feature_df=rfm_df,
    desc="RFM features for capstone",
    timestamp_col="_FEATURE_TIMESTAMP",
)
rfm_fv = register_feature_view(fs, rfm_fv, version="V1", overwrite=True)

# Step 3b: Create and register behavioral feature view
behavioral_fv = create_external_feature_view(
    name="CAPSTONE_BEHAVIORAL_FV",
    entities=[customer_entity],
    feature_df=behavioral_df,
    desc="Behavioral features for capstone",
    timestamp_col="_FEATURE_TIMESTAMP",
)
behavioral_fv = register_feature_view(fs, behavioral_fv, version="V1", overwrite=True)

print("Feature Store ready: entity + 2 feature views registered.")

Feature Store ready: entity + 2 feature views registered.


### Solution B4: Generate Training Set

> **Key Concept**: A **spine DataFrame** defines *which* entities and *when*
> to retrieve features for. The Feature Store then performs a point-in-time
> join across all requested feature views, producing a wide training table.

This pattern prevents **data leakage** — the timestamp on the spine ensures
you only join features that were available at the time of each observation.
In production, the spine is typically your label table (e.g., a table of
known outcomes keyed by customer and date).

In [9]:
# TODO B4: Generate a training set from the Feature Store
#   1. Create a spine DataFrame from CUSTOMER_RFM_FEATURES
#   2. Retrieve the registered feature views
#   3. Generate the training set by joining spine + both feature views
#
# Hint: See notebook 09, section 9.2 and notebook 12, cell 6
#   - create_spine_dataframe(session, entity_table=..., entity_key='CUSTOMER_ID')
#   - get_feature_view(fs, name, version)
#   - generate_training_set(fs, spine_df, features=[...], name=...)

# Step 1: Create spine
spine_df = create_spine_dataframe(session, entity_table="CUSTOMER_RFM_FEATURES", entity_key="CUSTOMER_ID")  # SOLUTION

# Step 2: Get registered feature views
rfm_fv = get_feature_view(fs, "CAPSTONE_RFM_FV", "V1")  # SOLUTION
behavioral_fv = get_feature_view(fs, "CAPSTONE_BEHAVIORAL_FV", "V1")  # SOLUTION

# Step 3: Generate training set
training_set = generate_training_set(fs, spine_df, features=[rfm_fv, behavioral_fv], name="CAPSTONE_TRAINING")  # SOLUTION

training_df = training_set.read.to_snowpark_dataframe()
print(f"Training set: {training_df.count()} rows, {len(training_df.columns)} columns")
print(f"Columns: {training_df.columns}")

Training set: 2000 rows, 23 columns
Columns: ['CUSTOMER_ID', 'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D', 'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL', 'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER', 'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS', 'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D', 'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'PREFERRED_CHANNEL', 'AVG_DURATION_SECONDS']


In [10]:
# --- Validation B ---
assert rfm_df.count() > 0, "B1: rfm_df should have rows"
assert behavioral_df.count() > 0, "B2: behavioral_df should have rows"
assert training_df.count() > 0, "B4: training_df should have rows"
assert len(training_df.columns) >= 20, "B4: training set should have 20+ columns"
assert 'CUSTOMER_ID' in training_df.columns, "B4: training set should have CUSTOMER_ID"
print("Part B: All validations passed!")

Part B: All validations passed!


---

## Part C: Label Creation & Model Training (15 min)

**Skills**: Label engineering, model training, evaluation, comparison (Weeks 1 + 2)

The target variable is `AT_RISK`: customers with **high support activity**,
**declining engagement**, and **few recent orders** are labeled as at-risk.

This is different from the churn label you built in notebook 12, which used
recency and order count. Here, support tickets and interaction patterns
are the primary signals.

You will train **3 model types** and compare their performance.

In [11]:
# --- Label creation (given — just run this cell) ---
# Risk label: support issues + low engagement + noise for realistic separation

FEATURE_COLS = [
    'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
    'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
    'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
    'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
    'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
    'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
]
LABEL_COL = 'AT_RISK'

labeled_df = (
    training_df
    .with_column("_NOISE", F.uniform(0.0, 1.0, F.random(99)))
    .with_column(
        LABEL_COL,
        F.when(
            # High-risk: frequent support + disengaged + few orders (90% rate)
            (F.col("SUPPORT_TICKETS_30D") >= 2)
            & (F.col("DAYS_SINCE_LAST_INTERACTION") > 20)
            & (F.col("ORDERS_30D") <= 1)
            & (F.col("_NOISE") < F.lit(0.90)),
            F.lit(1)
        ).when(
            # Medium-risk: low engagement + low spend (20% rate)
            (F.col("INTERACTIONS_30D") < 3)
            & (F.col("SPEND_30D") < F.lit(500))
            & (F.col("_NOISE") < F.lit(0.20)),
            F.lit(1)
        ).when(
            # Baseline noise: 3% of other customers
            F.col("_NOISE") < F.lit(0.03),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .drop("_NOISE")
)

train_df, test_df = split_data(labeled_df.select(FEATURE_COLS + [LABEL_COL]))
print(f"Train: {train_df.count()}, Test: {test_df.count()}")

Train: 1569, Test: 431


### Solutions C1–C3: Train Three Model Types

> **Key Concept**: Training multiple model types on the same data is a
> standard **model selection** practice. Each algorithm makes different
> assumptions about the data:
>
> - **XGBoost**: Gradient-boosted trees — handles non-linear relationships
>   and feature interactions automatically. Usually the strongest performer
>   on tabular data.
> - **Random Forest**: Bagged decision trees — more robust to overfitting
>   than a single tree, good baseline for comparison.
> - **Logistic Regression**: Linear model — fast, interpretable, and serves
>   as the "is a complex model even necessary?" sanity check.

The `train_model()` wrapper standardizes the interface across all three.
Note that `model_params` are passed through to the underlying scikit-learn
or XGBoost estimator, so you can tune hyperparameters without changing
the calling pattern.

For logistic regression, `max_iter=1000` ensures convergence on larger
feature sets — the default of 100 often warns about non-convergence.

In [12]:
# TODO C1: Train an XGBoost model
# Hint: See notebook 03, section 3.2 and notebook 10, cell 7
#   - train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='xgboost',
#       model_params={'n_estimators': 150, 'max_depth': 7})

model_xgb = train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='xgboost', model_params={'n_estimators': 150, 'max_depth': 7})  # SOLUTION

preds_xgb = predict(model_xgb, test_df)
metrics_xgb = evaluate_binary_classifier(preds_xgb, LABEL_COL, 'PREDICTION')
print("XGBoost:", metrics_xgb)

XGBoost: {'accuracy': 0.946636, 'precision': 0.5, 'recall': 0.043478260869565216, 'f1_score': 0.08}


In [13]:
# TODO C2: Train a Random Forest model
# Hint: Same pattern as C1, but model_type='random_forest'
#   - Use model_params={'n_estimators': 150, 'max_depth': 7}

model_rf = train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='random_forest', model_params={'n_estimators': 150, 'max_depth': 7})  # SOLUTION

preds_rf = predict(model_rf, test_df)
metrics_rf = evaluate_binary_classifier(preds_rf, LABEL_COL, 'PREDICTION')
print("Random Forest:", metrics_rf)

Random Forest: {'accuracy': 0.946636, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}


In [14]:
# TODO C3: Train a Logistic Regression model
# Hint: Same pattern, but model_type='logistic_regression'
#   - Use model_params={'max_iter': 1000}

model_lr = train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='logistic_regression', model_params={'max_iter': 1000})  # SOLUTION

preds_lr = predict(model_lr, test_df)
metrics_lr = evaluate_binary_classifier(preds_lr, LABEL_COL, 'PREDICTION')
print("Logistic Regression:", metrics_lr)

Logistic Regression: {'accuracy': 0.946636, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}


In [15]:
# --- Compare all 3 models side-by-side ---
import pandas as pd

comparison_df = pd.DataFrame([
    {'model': 'XGBoost', **metrics_xgb},
    {'model': 'Random Forest', **metrics_rf},
    {'model': 'Logistic Regression', **metrics_lr},
]).set_index('model')

print(comparison_df.to_string())
print(f"\nBest model by F1: {comparison_df['f1_score'].idxmax()}")

                     accuracy  precision    recall  f1_score
model                                                       
XGBoost              0.946636        0.5  0.043478      0.08
Random Forest        0.946636        0.0  0.000000      0.00
Logistic Regression  0.946636        0.0  0.000000      0.00

Best model by F1: XGBoost


In [16]:
# --- Validation C ---
for name, m in [('XGBoost', metrics_xgb), ('RF', metrics_rf), ('LR', metrics_lr)]:
    assert isinstance(m, dict), f"C: {name} metrics should be a dict"
    assert set(m.keys()) == {'accuracy', 'precision', 'recall', 'f1_score'}, \
        f"C: {name} metrics should have 4 keys"
    assert m['accuracy'] < 1.0, f"C: {name} accuracy is 1.0 — something is wrong"
print("Part C: All validations passed!")

Part C: All validations passed!


---

## Part D: Model Registry & Versioning (10 min)

**Skills**: Model registration, version comparison, alias promotion (Week 3)

Register all 3 models as versions of `RISK_SCORER_CAPSTONE`,
compare their metrics, and promote the best one to production.

In [17]:
# --- Registry setup (given) ---
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = 'RISK_SCORER_CAPSTONE'
sample_input = test_df.select(FEATURE_COLS).limit(10)

# Idempotent cleanup
try:
    delete_model(registry, MODEL_NAME)
    print(f"Deleted existing {MODEL_NAME} (re-run cleanup)")
except Exception:
    pass

print("Registry ready.")

Registry ready.


### Solution D1: Register Model Versions

> **Key Concept**: The Model Registry stores each model as an **immutable
> version** with its serialized artifact, sample input schema, and metrics.
> Immutability means you can always reproduce or roll back — a version once
> logged never changes.

The critical detail here is the **conversion method**:
- XGBoost models use `.to_xgboost()` to get the native `xgb.Booster`
- scikit-learn models (Random Forest, Logistic Regression) use `.to_sklearn()`

Getting this wrong causes a serialization error at registration time.
The `sample_input` defines the expected input schema — Snowflake uses it
to validate inference calls and generate the SQL function signature.

In [18]:
# TODO D1: Register all 3 models as versions V1, V2, V3
# Hint: See notebook 10, cells 10–11
#   - log_model(registry=registry, model=model_xgb.to_xgboost(),
#       model_name=MODEL_NAME, version_name='V1',
#       sample_input=sample_input, metrics=metrics_xgb)
#   - For RF: model_rf.to_sklearn(), for LR: model_lr.to_sklearn()

# V1: XGBoost
log_model(registry=registry, model=model_xgb.to_xgboost(), model_name=MODEL_NAME, version_name='V1', sample_input=sample_input, metrics=metrics_xgb)  # SOLUTION
print(f"Registered {MODEL_NAME} V1 (XGBoost)")

# V2: Random Forest
log_model(registry=registry, model=model_rf.to_sklearn(), model_name=MODEL_NAME, version_name='V2', sample_input=sample_input, metrics=metrics_rf)  # SOLUTION
print(f"Registered {MODEL_NAME} V2 (Random Forest)")

# V3: Logistic Regression
log_model(registry=registry, model=model_lr.to_sklearn(), model_name=MODEL_NAME, version_name='V3', sample_input=sample_input, metrics=metrics_lr)  # SOLUTION
print(f"Registered {MODEL_NAME} V3 (Logistic Regression)")

Model logged successfully.: 100%|██████████| 6/6 [00:22<00:00,  3.82s/it]                          
Registered RISK_SCORER_CAPSTONE V1 (XGBoost)
Model logged successfully.: 100%|██████████| 6/6 [00:21<00:00,  3.51s/it]                          
Registered RISK_SCORER_CAPSTONE V2 (Random Forest)
Model logged successfully.: 100%|██████████| 6/6 [00:19<00:00,  3.18s/it]                          
Registered RISK_SCORER_CAPSTONE V3 (Logistic Regression)


### Solution D2: Compare Versions

> **Key Concept**: `compare_model_versions()` retrieves the metrics stored
> at registration time for each version. This is how you do **offline model
> selection** — compare all candidates on the same test set before promoting
> any to production.

In production workflows, this comparison step is often automated in CI/CD:
a pipeline trains a new version, compares it against the current production
version, and only promotes if the new one meets a quality gate (e.g.,
F1 >= current F1 - 0.02).

In [19]:
# TODO D2: Compare metrics across all 3 versions
# Hint: See notebook 10, section 10.5
#   - compare_model_versions(registry, MODEL_NAME, ['V1', 'V2', 'V3'])

comparison = compare_model_versions(registry, MODEL_NAME, ['V1', 'V2', 'V3'])  # SOLUTION

# Display side-by-side
rows = []
for item in comparison:
    row = {'version': item['version']}
    row.update(item['metrics'])
    rows.append(row)

registry_comparison = pd.DataFrame(rows).set_index('version')
print(registry_comparison.to_string())

         accuracy  precision    recall  f1_score
version                                         
V1       0.946636        0.5  0.043478      0.08
V2       0.946636        0.0  0.000000      0.00
V3       0.946636        0.0  0.000000      0.00


### Solution D3: Promote to Production

> **Key Concept**: **Alias-based promotion** is a zero-downtime deployment
> pattern. Instead of swapping model files or redeploying services, you move
> the `production` alias from one version to another. All consumers that
> reference the alias automatically pick up the new version on their next call.

The three-step promotion pattern:
1. **`set_default_version`** — determines which version is used when no
   version or alias is specified
2. **`set_model_alias`** — attaches a named pointer (e.g., `production`)
   that consumers reference
3. **`set_model_comment`** — documents *why* this version was promoted,
   creating an audit trail

In [20]:
# TODO D3: Promote the best model to production
#   1. Identify the version with the highest F1 score from the comparison
#   2. Set it as the default version
#   3. Set the 'production' alias on it
#   4. Add a comment explaining why this version was promoted
#
# Hint: See notebook 10, section 10.6 and notebook 11, sections 11.2 + 11.6
#   - set_default_version(registry, MODEL_NAME, version_name)
#   - set_model_alias(registry, MODEL_NAME, version_name, 'production')
#   - set_model_comment(registry, MODEL_NAME, 'your comment', version_name=...)

# Step 1: Find best version by F1
best_version = registry_comparison['f1_score'].idxmax()
print(f"Best version: {best_version} (F1={registry_comparison.loc[best_version, 'f1_score']:.4f})")

# Step 2: Set default
set_default_version(registry, MODEL_NAME, best_version)  # SOLUTION

# Step 3: Set production alias
set_model_alias(registry, MODEL_NAME, best_version, 'production')  # SOLUTION

# Step 4: Add comment
set_model_comment(registry, MODEL_NAME, f"Promoted {best_version} to production — best F1 score in capstone evaluation", version_name=best_version)  # SOLUTION

# Verify
versions_df = list_versions(registry, MODEL_NAME)
print(versions_df[['name', 'aliases']].to_string())

Best version: V1 (F1=0.0800)
  name                           aliases
0   V1  ["PRODUCTION","DEFAULT","FIRST"]
1   V2                                []
2   V3                          ["LAST"]


In [21]:
# --- Validation D ---
versions_df = list_versions(registry, MODEL_NAME)
assert len(versions_df) == 3, "D1: Should have 3 registered versions"

all_aliases = ' '.join(versions_df['aliases'].tolist())
assert 'PRODUCTION' in all_aliases, "D3: One version should have the PRODUCTION alias"
print("Part D: All validations passed!")

Part D: All validations passed!


---

## Part E: Production Deployment (10 min)

**Skills**: Inference by alias, SQL inference, prediction monitoring (Week 3)

Deploy the production model for scoring and monitor the output.

### Solution E1: Inference by Alias

> **Key Concept**: Loading a model by **alias** rather than version number
> decouples consumers from the release process. The scoring pipeline always
> calls `get_model_by_alias(... 'production')` — when a new version is
> promoted, consumers pick it up automatically with zero code changes.

The `.run()` method executes the model's `predict` function server-side.
The input DataFrame's columns must match the `sample_input` schema that was
provided at registration time.

In [22]:
# TODO E1: Run inference using the production alias
# Hint: See notebook 11, section 11.3 and notebook 12, cell 13
#   - get_model_by_alias(registry, MODEL_NAME, 'production')
#   - production_model.run(input_df, function_name='predict')

# Load the production model by alias
production_model = get_model_by_alias(registry, MODEL_NAME, 'production')  # SOLUTION

# Score the test set
predictions = production_model.run(test_df.select(FEATURE_COLS), function_name='predict')  # SOLUTION

print("Predictions from production model:")
predictions.show(5)

Predictions from production model:
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"   |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"AVG_DURATION_SECONDS"  |"output_feature_0"  |
-----------------------------------------------------------------------

In [23]:
# TODO E2: Write a SQL inference query using MODEL!PREDICT()
# Hint: See notebook 13, section 13.2
#   - Use a CTE to select the 21 feature columns from the joined tables
#   - Then SELECT *, MODEL_NAME!PREDICT(*) AS PREDICTION FROM features
#   - Replace MODEL_NAME with the actual fully-qualified model name
#     ({current_db}.{current_schema}.RISK_SCORER_CAPSTONE)

sql_predictions = session.sql(f"""
    WITH features AS (
        SELECT
            DAYS_SINCE_LAST_ORDER, ORDERS_30D, ORDERS_90D, ORDERS_365D,
            ORDERS_TOTAL, SPEND_30D, SPEND_90D, SPEND_365D, SPEND_TOTAL,
            AVG_ORDER_VALUE, TOTAL_ITEMS, AVG_ITEMS_PER_ORDER,
            TOTAL_PAGE_VIEWS, TOTAL_CLICKS, TOTAL_SUPPORT_TICKETS,
            TOTAL_EMAIL_ENGAGEMENTS, INTERACTIONS_30D, SUPPORT_TICKETS_30D,
            INTERACTIONS_90D, DAYS_SINCE_LAST_INTERACTION, AVG_DURATION_SECONDS
        FROM {current_db}.{current_schema}.CUSTOMER_RFM_FEATURES r
        JOIN {current_db}.{current_schema}.CUSTOMER_BEHAVIORAL_FEATURES b
            ON r.CUSTOMER_ID = b.CUSTOMER_ID
        LIMIT 10
    )
    SELECT *, {current_db}.{current_schema}.RISK_SCORER_CAPSTONE!PREDICT(*) AS PREDICTION
    FROM features
""")  # SOLUTION

print("SQL inference results:")
sql_predictions.show()

SQL inference results:
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"AVG_DURATION_SECONDS"  |"PREDICTION"             |
---------------------------------------------------------------------------

In [24]:
# TODO E3: Score all customers and check the prediction distribution
# Hint: See notebook 13, sections 13.3 and 13.5
#   1. Use retrieve_feature_values() to get features for all customers
#   2. Score with the production model
#   3. Save to a table and query the distribution

# Retrieve features for all customers
all_features = retrieve_feature_values(
    fs=fs,
    spine_df=spine_df,
    features=[rfm_fv, behavioral_fv],
)

# Score
all_predictions = production_model.run(
    all_features.select(FEATURE_COLS),
    function_name='predict',
)

# Save to table
output_table = f"{current_db}.{current_schema}.CAPSTONE_RISK_PREDICTIONS"
all_predictions.write.mode('overwrite').save_as_table(output_table)

# TODO: Query prediction distribution
# Hint: See notebook 13, cell 12
#   - Group by "output_feature_0", count, and compute percentage

distribution = session.sql(f"""
    SELECT
        "output_feature_0" AS PREDICTION,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS PERCENTAGE
    FROM {output_table}
    GROUP BY "output_feature_0"
    ORDER BY "output_feature_0"
""")

print("Prediction distribution:")
distribution.show()

Prediction distribution:
-----------------------------------------
|"PREDICTION"  |"COUNT"  |"PERCENTAGE"  |
-----------------------------------------
|0             |1922     |96.10         |
|1             |78       |3.90          |
-----------------------------------------



In [25]:
# TODO E4: Compare production predictions with registered training metrics
# Hint: See notebook 13, cell 13
#   - get_model_metrics(registry, MODEL_NAME, best_version)

training_metrics = get_model_metrics(registry, MODEL_NAME, best_version)  # SOLUTION

print("Registered training metrics:")
for metric, value in training_metrics.items():
    print(f"  {metric}: {value}")

pred_count = session.table(output_table).count()
print(f"\nTotal customers scored: {pred_count}")
print("\nCompare prediction distribution against training class balance.")
print("If they diverge significantly, investigate feature drift.")

Registered training metrics:
  accuracy: 0.946636
  precision: 0.5
  recall: 0.043478260869565216
  f1_score: 0.08

Total customers scored: 2000

Compare prediction distribution against training class balance.
If they diverge significantly, investigate feature drift.


In [26]:
# --- Validation E ---
assert predictions.count() > 0, "E1: predictions should have rows"
assert session.table(output_table).count() > 0, "E3: predictions table should exist"
assert isinstance(training_metrics, dict), "E4: training_metrics should be a dict"
print("Part E: All validations passed!")
print("")
print("=" * 60)
print("CAPSTONE CHALLENGE COMPLETE!")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Best version: {best_version}")
print(f"Production alias: set")
print(f"Customers scored: {session.table(output_table).count()}")
print(f"Training F1: {training_metrics.get('f1_score', 'N/A')}")

Part E: All validations passed!

CAPSTONE CHALLENGE COMPLETE!

Model: RISK_SCORER_CAPSTONE
Best version: V1
Production alias: set
Customers scored: 2000
Training F1: 0.08


---

## Cleanup

Uncomment to remove all capstone resources.

In [27]:
# delete_model(registry, MODEL_NAME)
# session.sql(f"DROP TABLE IF EXISTS {output_table}").collect()
# print("Capstone resources cleaned up.")

In [28]:
session.close()